# GM2$^-$ Mode 3 Sign Convention: SMODES vs. Notebook

This notebook demonstrates why `atomic_positions.ipynb` negates the raw SMODES output for $GM2^-$ mode 3, and proves that both conventions are physically equivalent (related by $+Q \leftrightarrow -Q$).

**Strategy:**
1. Start from SMODES reference positions (no wrapping, ever)
2. Apply A1g shift to reach equilibrium
3. Apply $GM2^-$ mode 3 in the **raw SMODES convention** ($+Q$)
4. Apply $GM2^-$ mode 3 in the **notebook convention** ($-Q$)
5. Show the two are related by a symmetry operation (inversion) and give identical physics

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

a = 10.19  # cubic lattice constant (Angstrom)
A_prim = np.array([[0, a/2, a/2], [a/2, 0, a/2], [a/2, a/2, 0]])


def min_image_distance(r1, r2):
    """Minimum image distance in the primitive cell."""
    d = r2 - r1
    s = np.linalg.solve(A_prim.T, d)
    s -= np.round(s)
    return np.linalg.norm(A_prim.T @ s)

## 1. SMODES Reference Positions

These are the positions SMODES prints for the GM2$^-$ irrep. The 48f oxygens are at the high-symmetry parent ($x=0$), not the physical equilibrium.

In [2]:
# SMODES reference positions (Cartesian, Angstrom)
R_ref = {
     1: np.array([ 5.09500,  5.09500,  5.09500]),  # Y
     2: np.array([ 5.09500,  2.54750,  2.54750]),  # Y
     3: np.array([ 2.54750,  5.09500,  2.54750]),  # Y
     4: np.array([ 2.54750,  2.54750,  5.09500]),  # Y
     5: np.array([ 0.00000,  0.00000,  0.00000]),  # Ir
     6: np.array([ 0.00000,  2.54750,  2.54750]),  # Ir
     7: np.array([ 2.54750,  0.00000,  2.54750]),  # Ir
     8: np.array([ 2.54750,  2.54750,  0.00000]),  # Ir
     9: np.array([ 3.82125,  3.82125,  3.82125]),  # O (8b)
    10: np.array([ 6.36875,  6.36875,  6.36875]),  # O (8b)
    11: np.array([ 0.00000,  1.27375,  1.27375]),  # O (48f)
    12: np.array([ 2.54750,  1.27375,  1.27375]),  # O (48f)
    13: np.array([ 1.27375,  1.27375,  0.00000]),  # O (48f)
    14: np.array([ 1.27375,  1.27375,  2.54750]),  # O (48f)
    15: np.array([ 1.27375,  0.00000,  1.27375]),  # O (48f)
    16: np.array([ 1.27375,  2.54750,  1.27375]),  # O (48f)
    17: np.array([ 3.82125,  0.00000,  3.82125]),  # O (48f)
    18: np.array([ 3.82125,  7.64250,  3.82125]),  # O (48f)
    19: np.array([ 0.00000,  3.82125,  3.82125]),  # O (48f)
    20: np.array([ 7.64250,  3.82125,  3.82125]),  # O (48f)
    21: np.array([ 3.82125,  3.82125,  0.00000]),  # O (48f)
    22: np.array([ 3.82125,  3.82125,  7.64250]),  # O (48f)
}

species = {i: 'Y' for i in range(1,5)} | {i: 'Ir' for i in range(5,9)} | {i: 'O' for i in range(9,23)}

print('SMODES reference positions (atoms 11-22 are at x=0 parent):')
for i in sorted(R_ref):
    print(f'  {i:2d} {species[i]:2s}  {R_ref[i]}')

SMODES reference positions (atoms 11-22 are at x=0 parent):
   1 Y   [5.095 5.095 5.095]
   2 Y   [5.095  2.5475 2.5475]
   3 Y   [2.5475 5.095  2.5475]
   4 Y   [2.5475 2.5475 5.095 ]
   5 Ir  [0. 0. 0.]
   6 Ir  [0.     2.5475 2.5475]
   7 Ir  [2.5475 0.     2.5475]
   8 Ir  [2.5475 2.5475 0.    ]
   9 O   [3.82125 3.82125 3.82125]
  10 O   [6.36875 6.36875 6.36875]
  11 O   [0.      1.27375 1.27375]
  12 O   [2.5475  1.27375 1.27375]
  13 O   [1.27375 1.27375 0.     ]
  14 O   [1.27375 1.27375 2.5475 ]
  15 O   [1.27375 0.      1.27375]
  16 O   [1.27375 2.5475  1.27375]
  17 O   [3.82125 0.      3.82125]
  18 O   [3.82125 7.6425  3.82125]
  19 O   [0.      3.82125 3.82125]
  20 O   [7.6425  3.82125 3.82125]
  21 O   [3.82125 3.82125 0.     ]
  22 O   [3.82125 3.82125 7.6425 ]


## 2. GM1+ (A1g) Displacement — From SMODES Output

This is the breathing mode, directly from `SMODES.Y2Ir2O7.out` (lines 49–62). It only acts on 48f oxygens.

In [3]:
# Raw SMODES output for GM1+ (A1g)
u_A1g = {
    11: np.array([+1,  0,  0]),
    12: np.array([-1,  0,  0]),
    13: np.array([ 0,  0, +1]),
    14: np.array([ 0,  0, -1]),
    15: np.array([ 0, +1,  0]),
    16: np.array([ 0, -1,  0]),
    17: np.array([ 0, -1,  0]),
    18: np.array([ 0, +1,  0]),
    19: np.array([-1,  0,  0]),
    20: np.array([+1,  0,  0]),
    21: np.array([ 0,  0, -1]),
    22: np.array([ 0,  0, +1]),
}

## 3. Equilibrium Positions (No Wrapping)

Apply A1g with amplitude $\delta = a \cdot x_{\text{OC2}} = 10.19 \times 0.337932 = 3.4435$ Å.

**We do NOT wrap.** Some coordinates go negative or exceed $a$. That's fine — QE and min-image handle periodicity. This keeps the atom labels in their natural positions.

In [4]:
x_OC2 = 0.337932
delta_A1g = x_OC2 * a
print(f'A1g amplitude: delta = x_OC2 * a = {delta_A1g:.6f} Angstrom')
print()

# Equilibrium = SMODES parent + A1g shift (no wrapping)
R_eq = {}
for i in sorted(R_ref):
    if i in u_A1g:
        R_eq[i] = R_ref[i] + delta_A1g * u_A1g[i]
    else:
        R_eq[i] = R_ref[i].copy()

print('Equilibrium positions (NO wrap):')
print(f'{"Atom":>4} {"Species":>4} {"x":>10} {"y":>10} {"z":>10}  Notes')
for i in sorted(R_eq):
    r = R_eq[i]
    note = ''
    if any(c < 0 for c in r) or any(c > a for c in r):
        note = '<-- outside [0, a]'
    print(f'{i:4d} {species[i]:>4}  {r[0]:10.5f} {r[1]:10.5f} {r[2]:10.5f}  {note}')

A1g amplitude: delta = x_OC2 * a = 3.443527 Angstrom

Equilibrium positions (NO wrap):
Atom Species          x          y          z  Notes
   1    Y     5.09500    5.09500    5.09500  
   2    Y     5.09500    2.54750    2.54750  
   3    Y     2.54750    5.09500    2.54750  
   4    Y     2.54750    2.54750    5.09500  
   5   Ir     0.00000    0.00000    0.00000  
   6   Ir     0.00000    2.54750    2.54750  
   7   Ir     2.54750    0.00000    2.54750  
   8   Ir     2.54750    2.54750    0.00000  
   9    O     3.82125    3.82125    3.82125  
  10    O     6.36875    6.36875    6.36875  
  11    O     3.44353    1.27375    1.27375  
  12    O    -0.89603    1.27375    1.27375  <-- outside [0, a]
  13    O     1.27375    1.27375    3.44353  
  14    O     1.27375    1.27375   -0.89603  <-- outside [0, a]
  15    O     1.27375    3.44353    1.27375  
  16    O     1.27375   -0.89603    1.27375  <-- outside [0, a]
  17    O     3.82125   -3.44353    3.82125  <-- outside [0, a]
  18  

In [5]:
# Verify: Ir-O(48f) distances at equilibrium
print('Nearest 6 Ir-O(48f) distances per Ir (expect ~2.01 Angstrom):')
for ir in [5, 6, 7, 8]:
    dists = sorted(min_image_distance(R_eq[ir], R_eq[o]) for o in range(11, 23))
    print(f'  Ir{ir}: {dists[0]:.4f}  {dists[1]:.4f}  {dists[2]:.4f}  {dists[3]:.4f}  {dists[4]:.4f}  {dists[5]:.4f}')

Nearest 6 Ir-O(48f) distances per Ir (expect ~2.01 Angstrom):
  Ir5: 2.0119  2.0119  2.0119  2.0119  2.0119  2.0119
  Ir6: 2.0119  2.0119  2.0119  2.0119  2.0119  2.0119
  Ir7: 2.0119  2.0119  2.0119  2.0119  2.0119  2.0119
  Ir8: 2.0119  2.0119  2.0119  2.0119  2.0119  2.0119


## 4. GM2$^-$ Mode Vectors — Raw SMODES Output

Copied directly from `SMODES.Y2Ir2O7.out` lines 259–280.

Note: mode 3 vectors on atoms 11–16 are **identical** to A1g, and **opposite** to A1g on atoms 17–22.

In [6]:
# GM2- mode 1 (Y sublattice) — raw SMODES
u_GM2_mode1_smodes = {
    1: np.array([+1, +1, +1]),
    2: np.array([+1, -1, -1]),
    3: np.array([-1, +1, -1]),
    4: np.array([-1, -1, +1]),
}

# GM2- mode 2 (Ir sublattice) — raw SMODES
u_GM2_mode2_smodes = {
    5: np.array([+1, +1, +1]),
    6: np.array([+1, -1, -1]),
    7: np.array([-1, +1, -1]),
    8: np.array([-1, -1, +1]),
}

# GM2- mode 3 (48f O sublattice) — raw SMODES
u_GM2_mode3_smodes = {
    11: np.array([+1,  0,  0]),
    12: np.array([-1,  0,  0]),
    13: np.array([ 0,  0, +1]),
    14: np.array([ 0,  0, -1]),
    15: np.array([ 0, +1,  0]),
    16: np.array([ 0, -1,  0]),
    17: np.array([ 0, +1,  0]),
    18: np.array([ 0, -1,  0]),
    19: np.array([+1,  0,  0]),
    20: np.array([-1,  0,  0]),
    21: np.array([ 0,  0, +1]),
    22: np.array([ 0,  0, -1]),
}

print('Comparison: A1g vs. raw SMODES GM2- mode 3')
print(f'{"Atom":>4}  {"A1g":>12}  {"GM2- m3":>12}  {"Relation":>10}')
for i in range(11, 23):
    a1 = u_A1g[i]
    m3 = u_GM2_mode3_smodes[i]
    if np.allclose(m3, a1):    rel = 'SAME'
    elif np.allclose(m3, -a1): rel = 'OPPOSITE'
    else:                       rel = '???'
    print(f'{i:4d}  {str(a1):>12}  {str(m3):>12}  {rel:>10}')

print()
print('Summary: atoms 11-16 match A1g, atoms 17-22 are opposite.')
print('This is one valid A2u phase choice (SMODES chose +Q).')

Comparison: A1g vs. raw SMODES GM2- mode 3
Atom           A1g       GM2- m3    Relation
  11       [1 0 0]       [1 0 0]        SAME
  12    [-1  0  0]    [-1  0  0]        SAME
  13       [0 0 1]       [0 0 1]        SAME
  14    [ 0  0 -1]    [ 0  0 -1]        SAME
  15       [0 1 0]       [0 1 0]        SAME
  16    [ 0 -1  0]    [ 0 -1  0]        SAME
  17    [ 0 -1  0]       [0 1 0]    OPPOSITE
  18       [0 1 0]    [ 0 -1  0]    OPPOSITE
  19    [-1  0  0]       [1 0 0]    OPPOSITE
  20       [1 0 0]    [-1  0  0]    OPPOSITE
  21    [ 0  0 -1]       [0 0 1]    OPPOSITE
  22       [0 0 1]    [ 0  0 -1]    OPPOSITE

Summary: atoms 11-16 match A1g, atoms 17-22 are opposite.
This is one valid A2u phase choice (SMODES chose +Q).


## 5. Apply $GM2^-$ Mode 3 in Raw SMODES Convention ($+Q$)

Displace the equilibrium positions by $\delta = 0.01$ Å along the SMODES mode 3 vectors.

Physical check: compute how the Ir–O distances change.

In [7]:
Q = 0.01  # Angstrom

# Displaced positions: +Q SMODES convention
R_plusQ = {}
for i in sorted(R_eq):
    d = u_GM2_mode3_smodes.get(i, np.zeros(3))
    R_plusQ[i] = R_eq[i] + Q * d

print(f'Displaced positions (+Q = +{Q} Angstrom, SMODES convention):')
print(f'{"Atom":>4} {"Equil. position":>34} {"Displacement":>14} {"Displaced position":>34}')
for i in range(11, 23):
    d = u_GM2_mode3_smodes.get(i, np.zeros(3))
    print(f'{i:4d}  {R_eq[i]}   {Q*d}   {R_plusQ[i]}')

Displaced positions (+Q = +0.01 Angstrom, SMODES convention):
Atom                    Equil. position   Displacement                 Displaced position
  11  [3.443527 1.27375  1.27375 ]   [0.01 0.   0.  ]   [3.453527 1.27375  1.27375 ]
  12  [-0.896027  1.27375   1.27375 ]   [-0.01  0.    0.  ]   [-0.906027  1.27375   1.27375 ]
  13  [1.27375  1.27375  3.443527]   [0.   0.   0.01]   [1.27375  1.27375  3.453527]
  14  [ 1.27375   1.27375  -0.896027]   [ 0.    0.   -0.01]   [ 1.27375   1.27375  -0.906027]
  15  [1.27375  3.443527 1.27375 ]   [0.   0.01 0.  ]   [1.27375  3.453527 1.27375 ]
  16  [ 1.27375  -0.896027  1.27375 ]   [ 0.   -0.01  0.  ]   [ 1.27375  -0.906027  1.27375 ]
  17  [ 3.82125  -3.443527  3.82125 ]   [0.   0.01 0.  ]   [ 3.82125  -3.433527  3.82125 ]
  18  [ 3.82125  11.086027  3.82125 ]   [ 0.   -0.01  0.  ]   [ 3.82125  11.076027  3.82125 ]
  19  [-3.443527  3.82125   3.82125 ]   [0.01 0.   0.  ]   [-3.433527  3.82125   3.82125 ]
  20  [11.086027  3.82125   3.82125

In [8]:
# Ir-O distances at +Q
print('Ir-O(48f) nearest 6 distances at +Q (SMODES convention):')
for ir in [5, 6, 7, 8]:
    dists = sorted(min_image_distance(R_plusQ[ir], R_plusQ[o]) for o in range(11, 23))[:6]
    print(f'  Ir{ir}: ' + '  '.join(f'{d:.5f}' for d in dists))

print()
print('Ir-O(48f) nearest 6 distances at equilibrium (for comparison):')
for ir in [5, 6, 7, 8]:
    dists = sorted(min_image_distance(R_eq[ir], R_eq[o]) for o in range(11, 23))[:6]
    print(f'  Ir{ir}: ' + '  '.join(f'{d:.5f}' for d in dists))

print()
print('The +Q SMODES convention is physically valid — it produces a legal distortion.')

Ir-O(48f) nearest 6 distances at +Q (SMODES convention):
  Ir5: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir6: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir7: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir8: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637

Ir-O(48f) nearest 6 distances at equilibrium (for comparison):
  Ir5: 2.01190  2.01190  2.01190  2.01190  2.01190  2.01190
  Ir6: 2.01190  2.01190  2.01190  2.01190  2.01190  2.01190
  Ir7: 2.01190  2.01190  2.01190  2.01190  2.01190  2.01190
  Ir8: 2.01190  2.01190  2.01190  2.01190  2.01190  2.01190

The +Q SMODES convention is physically valid — it produces a legal distortion.


## 6. Apply GM2- Mode 3 in Notebook Convention ($-Q$)

The `atomic_positions.ipynb` notebook negates all mode 3 vectors. This is the $-Q$ branch.

In [9]:
# GM2- mode 3 as used in atomic_positions.ipynb (= -SMODES)
u_GM2_mode3_notebook = {i: -v for i, v in u_GM2_mode3_smodes.items()}

print('Notebook convention = negated SMODES:')
print(f'{"Atom":>4}  {"SMODES (+Q)":>12}  {"Notebook (-Q)":>14}')
for i in range(11, 23):
    print(f'{i:4d}  {str(u_GM2_mode3_smodes[i]):>12}  {str(u_GM2_mode3_notebook[i]):>14}')

Notebook convention = negated SMODES:
Atom   SMODES (+Q)   Notebook (-Q)
  11       [1 0 0]      [-1  0  0]
  12    [-1  0  0]         [1 0 0]
  13       [0 0 1]      [ 0  0 -1]
  14    [ 0  0 -1]         [0 0 1]
  15       [0 1 0]      [ 0 -1  0]
  16    [ 0 -1  0]         [0 1 0]
  17       [0 1 0]      [ 0 -1  0]
  18    [ 0 -1  0]         [0 1 0]
  19       [1 0 0]      [-1  0  0]
  20    [-1  0  0]         [1 0 0]
  21       [0 0 1]      [ 0  0 -1]
  22    [ 0  0 -1]         [0 0 1]


In [10]:
# Displaced positions: -Q (notebook convention)
R_minusQ = {}
for i in sorted(R_eq):
    d = u_GM2_mode3_notebook.get(i, np.zeros(3))
    R_minusQ[i] = R_eq[i] + Q * d

print(f'Displaced positions (notebook convention, amplitude = {Q} Angstrom):')
print(f'{"Atom":>4} {"Equil. position":>34} {"Displacement":>14} {"Displaced position":>34}')
for i in range(11, 23):
    d = u_GM2_mode3_notebook.get(i, np.zeros(3))
    print(f'{i:4d}  {R_eq[i]}   {Q*d}   {R_minusQ[i]}')

Displaced positions (notebook convention, amplitude = 0.01 Angstrom):
Atom                    Equil. position   Displacement                 Displaced position
  11  [3.443527 1.27375  1.27375 ]   [-0.01  0.    0.  ]   [3.433527 1.27375  1.27375 ]
  12  [-0.896027  1.27375   1.27375 ]   [0.01 0.   0.  ]   [-0.886027  1.27375   1.27375 ]
  13  [1.27375  1.27375  3.443527]   [ 0.    0.   -0.01]   [1.27375  1.27375  3.433527]
  14  [ 1.27375   1.27375  -0.896027]   [0.   0.   0.01]   [ 1.27375   1.27375  -0.886027]
  15  [1.27375  3.443527 1.27375 ]   [ 0.   -0.01  0.  ]   [1.27375  3.433527 1.27375 ]
  16  [ 1.27375  -0.896027  1.27375 ]   [0.   0.01 0.  ]   [ 1.27375  -0.886027  1.27375 ]
  17  [ 3.82125  -3.443527  3.82125 ]   [ 0.   -0.01  0.  ]   [ 3.82125  -3.453527  3.82125 ]
  18  [ 3.82125  11.086027  3.82125 ]   [0.   0.01 0.  ]   [ 3.82125  11.096027  3.82125 ]
  19  [-3.443527  3.82125   3.82125 ]   [-0.01  0.    0.  ]   [-3.453527  3.82125   3.82125 ]
  20  [11.086027  3.8212

In [11]:
# Ir-O distances at -Q
print('Ir-O(48f) nearest 6 distances at -Q (notebook convention):')
for ir in [5, 6, 7, 8]:
    dists = sorted(min_image_distance(R_minusQ[ir], R_minusQ[o]) for o in range(11, 23))[:6]
    print(f'  Ir{ir}: ' + '  '.join(f'{d:.5f}' for d in dists))

print()
print('Compare with +Q:')
for ir in [5, 6, 7, 8]:
    dists = sorted(min_image_distance(R_plusQ[ir], R_plusQ[o]) for o in range(11, 23))[:6]
    print(f'  Ir{ir}: ' + '  '.join(f'{d:.5f}' for d in dists))

print()
print('The distances are identical — +Q and -Q produce the same magnitude of distortion.')

Ir-O(48f) nearest 6 distances at -Q (notebook convention):
  Ir5: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir6: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir7: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir8: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637

Compare with +Q:
  Ir5: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir6: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir7: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637
  Ir8: 2.00747  2.00747  2.00747  2.01637  2.01637  2.01637

The distances are identical — +Q and -Q produce the same magnitude of distortion.


## 7. Why Are They Equivalent? The Inversion Mapping

The $+Q$ and $-Q$ structures are related by **inversion through the origin**. Under inversion:
- Position $\mathbf{r} \to -\mathbf{r}$ (mod lattice)
- The crystal maps onto itself (Fd$\bar{3}$m has inversion symmetry)
- The energy is the same: $E(+Q) = E(-Q)$ for an odd-parity mode

So the frozen-phonon energy surface is symmetric: $E(Q) = E(-Q)$, and $\partial\theta/\partial Q$ computed at $+Q$ vs $-Q$ gives the same physical result (they're the same point on the potential energy surface, just relabeled by a symmetry operation).

In [12]:
# Demonstrate: inversion maps +Q structure to -Q structure
print('Verify: inversion maps +Q onto -Q')
print(f'{"Atom":>4}  {"R(+Q)":>30}  {"-R(+Q) = R(-Q)?":>30}  {"Match (min-image)":>10}')
print()

# Under inversion, atom i at +Q maps to its inversion partner at -Q.
# First find the inversion pairing:
inv_partner = {}
for i in range(11, 23):
    neg_r = -R_ref[i]  # inversion of SMODES reference position
    for j in range(11, 23):
        if min_image_distance(neg_r, R_ref[j]) < 0.01:
            inv_partner[i] = j
            break

print('Inversion pairing of 48f atoms:')
for i in range(11, 17):
    print(f'  atom {i} <--> atom {inv_partner[i]}')
print()

Verify: inversion maps +Q onto -Q
Atom                           R(+Q)                 -R(+Q) = R(-Q)?  Match (min-image)

Inversion pairing of 48f atoms:
  atom 11 <--> atom 19
  atom 12 <--> atom 20
  atom 13 <--> atom 21
  atom 14 <--> atom 22
  atom 15 <--> atom 17
  atom 16 <--> atom 18



In [13]:
# Verify: applying inversion to the +Q structure gives the -Q structure
print('Check that inversion(+Q structure) = -Q structure:')
print(f'{"i":>3} {"j=I(i)":>6}  {"R_i(+Q)":>28}  {"-R_i(+Q)":>28}  {"R_j(-Q)":>28}  {"d(Angstrom)":>10}')
max_err = 0
for i in range(11, 23):
    j = inv_partner[i]
    r_plus_i = R_plusQ[i]
    neg_r_plus_i = -r_plus_i    # inversion applied
    r_minus_j = R_minusQ[j]     # -Q structure at the partner atom
    d = min_image_distance(neg_r_plus_i, r_minus_j)
    max_err = max(max_err, d)
    print(f'{i:3d} {j:6d}  {r_plus_i}  {neg_r_plus_i}  {r_minus_j}  {d:.2e}')

print(f'\nMax discrepancy: {max_err:.2e} Angstrom')
print('=> The +Q and -Q structures are related by inversion. QED.')

Check that inversion(+Q structure) = -Q structure:
  i j=I(i)                       R_i(+Q)                      -R_i(+Q)                       R_j(-Q)  d(Angstrom)
 11     19  [3.453527 1.27375  1.27375 ]  [-3.453527 -1.27375  -1.27375 ]  [-3.453527  3.82125   3.82125 ]  4.00e-16
 12     20  [-0.906027  1.27375   1.27375 ]  [ 0.906027 -1.27375  -1.27375 ]  [11.096027  3.82125   3.82125 ]  2.26e-15
 13     21  [1.27375  1.27375  3.453527]  [-1.27375  -1.27375  -3.453527]  [ 3.82125   3.82125  -3.453527]  0.00e+00
 14     22  [ 1.27375   1.27375  -0.906027]  [-1.27375  -1.27375   0.906027]  [ 3.82125   3.82125  11.096027]  4.62e-15
 15     17  [1.27375  3.453527 1.27375 ]  [-1.27375  -3.453527 -1.27375 ]  [ 3.82125  -3.453527  3.82125 ]  4.00e-16
 16     18  [ 1.27375  -0.906027  1.27375 ]  [-1.27375   0.906027 -1.27375 ]  [ 3.82125  11.096027  3.82125 ]  2.26e-15
 17     15  [ 3.82125  -3.433527  3.82125 ]  [-3.82125   3.433527 -3.82125 ]  [1.27375  3.433527 1.27375 ]  4.00e-16
 18    

## 8. Physical Interpretation: Trigonal Distortion Within Each IrO₆ Octahedron

Each IrO₆ octahedron contains **3 atoms from Group A (11–16) and 3 from Group B (17–22)**. The GM2$^-$ mode 3 displacement is therefore NOT a breathing mode (which would move all 6 uniformly). Instead it is a **trigonal distortion**: within each octahedron, 3 bonds elongate while 3 shorten.

| Convention | Group A bonds (3 per Ir) | Group B bonds (3 per Ir) |
|------------|--------------------------|--------------------------|
| +Q (SMODES) | elongate | shorten |
| -Q (notebook) | shorten | elongate |

This is the key difference from A1g, which moves all 6 bonds in the same direction.

In [14]:
# Identify which 48f oxygens belong to each IrO6 octahedron
print('IrO6 octahedron membership:')
print(f'{"Ir":>4}  {"6 nearest O(48f)":>40}  {"Group A":>20}  {"Group B":>20}')
octahedra = {}
for ir in [5, 6, 7, 8]:
    dists = [(o, min_image_distance(R_eq[ir], R_eq[o])) for o in range(11, 23)]
    dists.sort(key=lambda x: x[1])
    members = [o for o, d in dists[:6]]
    octahedra[ir] = members
    grp_a = [o for o in members if o <= 16]
    grp_b = [o for o in members if o >= 17]
    print(f'  Ir{ir}  {members}  A: {grp_a}  B: {grp_b}')

print()
print('Each octahedron has exactly 3 Group-A and 3 Group-B atoms.')
print('The mode is a TRIGONAL distortion, not a breathing mode.')

# Compute Ir-O bond lengths at equilibrium, +Q, and -Q
print()
print('='*80)
print('Ir-O bond lengths: equilibrium vs +Q vs -Q')
print('='*80)
print()
print(f'{"Ir":>3} {"O":>3} {"Grp":>4} {"d(eq)":>8} {"d(+Q)":>8} {"d(-Q)":>8} {"Delta(+Q)":>10} {"Delta(-Q)":>10}')
for ir in [5, 6, 7, 8]:
    for o in octahedra[ir]:
        grp = 'A' if o <= 16 else 'B'
        d_eq = min_image_distance(R_eq[ir], R_eq[o])
        d_plus = min_image_distance(R_plusQ[ir], R_plusQ[o])
        d_minus = min_image_distance(R_minusQ[ir], R_minusQ[o])
        delta_plus = d_plus - d_eq
        delta_minus = d_minus - d_eq
        print(f'{ir:3d} {o:3d} {grp:>4} {d_eq:8.5f} {d_plus:8.5f} {d_minus:8.5f} {delta_plus:+10.5f} {delta_minus:+10.5f}')
    print()

print('Group A bonds ELONGATE under +Q (SMODES), SHORTEN under -Q (notebook).')
print('Group B bonds SHORTEN under +Q (SMODES), ELONGATE under -Q (notebook).')
print()
print('This is a trigonal distortion: 3 long + 3 short bonds within EACH octahedron.')

IrO6 octahedron membership:
  Ir                          6 nearest O(48f)               Group A               Group B
  Ir5  [12, 14, 16, 18, 20, 22]  A: [12, 14, 16]  B: [18, 20, 22]
  Ir6  [12, 13, 15, 21, 17, 20]  A: [12, 13, 15]  B: [21, 17, 20]
  Ir7  [11, 13, 16, 21, 19, 18]  A: [11, 13, 16]  B: [21, 19, 18]
  Ir8  [19, 11, 14, 15, 17, 22]  A: [11, 14, 15]  B: [19, 17, 22]

Each octahedron has exactly 3 Group-A and 3 Group-B atoms.
The mode is a TRIGONAL distortion, not a breathing mode.

Ir-O bond lengths: equilibrium vs +Q vs -Q

 Ir   O  Grp    d(eq)    d(+Q)    d(-Q)  Delta(+Q)  Delta(-Q)
  5  12    A  2.01190  2.01637  2.00747   +0.00447   -0.00443
  5  14    A  2.01190  2.01637  2.00747   +0.00447   -0.00443
  5  16    A  2.01190  2.01637  2.00747   +0.00447   -0.00443
  5  18    B  2.01190  2.00747  2.01637   -0.00443   +0.00447
  5  20    B  2.01190  2.00747  2.01637   -0.00443   +0.00447
  5  22    B  2.01190  2.00747  2.01637   -0.00443   +0.00447

  6  12    A  2.0119

## 9. Why Did the Notebook Choose $-Q$?

The notebook's comment says: *"Sign determined by the Ir-Ir bridge pair, NOT by the local A1g direction."*

Each 48f oxygen bridges two Ir atoms. In the AIAO pattern, these Ir atoms move in specific directions. The **$-Q$ convention** aligns each oxygen's displacement with the net force from its bridging Ir pair.

Let's verify: for each oxygen, compute the projection of its bridging Ir AIAO displacements onto the oxygen's displacement axis.

In [15]:
# AIAO pattern for Ir (from GM2- mode 2)
u_Ir_AIAO = {
    5: np.array([+1, +1, +1]),
    6: np.array([+1, -1, -1]),
    7: np.array([-1, +1, -1]),
    8: np.array([-1, -1, +1]),
}

# Find the two nearest Ir atoms for each 48f oxygen
print('Bridging Ir pair for each 48f oxygen and their AIAO projection:')
print(f'{"Atom":>4}  {"Bridge":>10}  {"Disp axis":>12}  {"Ir_a AIAO":>14}  {"Ir_b AIAO":>14}  '
      f'{"Proj_a":>6}  {"Proj_b":>6}  {"Net":>5}  {"SMODES":>8}  {"Notebook":>8}')

for i in range(11, 23):
    # Find two closest Ir
    ir_dists = [(ir, min_image_distance(R_eq[i], R_eq[ir])) for ir in [5,6,7,8]]
    ir_dists.sort(key=lambda x: x[1])
    ir_a, ir_b = ir_dists[0][0], ir_dists[1][0]
    
    # Displacement axis for this oxygen (nonzero component)
    axis = u_GM2_mode3_smodes[i]  # unit vector along displacement
    
    # Project AIAO vectors onto this axis
    proj_a = np.dot(u_Ir_AIAO[ir_a], axis)
    proj_b = np.dot(u_Ir_AIAO[ir_b], axis)
    net = proj_a + proj_b
    
    # What sign does SMODES use? What does notebook use?
    smodes_sign = '+' if np.dot(u_GM2_mode3_smodes[i], axis) > 0 else '-'
    notebook_sign = '+' if np.dot(u_GM2_mode3_notebook[i], axis) > 0 else '-'
    net_sign = '+' if net > 0 else '-'
    
    print(f'{i:4d}  Ir{ir_a}-Ir{ir_b}  {str(axis):>12}  {str(u_Ir_AIAO[ir_a]):>14}  '
          f'{str(u_Ir_AIAO[ir_b]):>14}  {proj_a:+5.0f}   {proj_b:+5.0f}   {net_sign:>3}   '
          f'{smodes_sign:>6}   {notebook_sign:>6}')

print()
print('The "Net" column shows the net AIAO projection along the oxygen axis.')
print('The notebook convention (-Q) matches the net Ir-bridge projection for every atom.')
print('The SMODES convention (+Q) is opposite to the Ir-bridge projection for atoms 11-16.')

Bridging Ir pair for each 48f oxygen and their AIAO projection:
Atom      Bridge     Disp axis       Ir_a AIAO       Ir_b AIAO  Proj_a  Proj_b    Net    SMODES  Notebook
  11  Ir7-Ir8       [1 0 0]      [-1  1 -1]      [-1 -1  1]     -1      -1     -        +        -
  12  Ir5-Ir6    [-1  0  0]         [1 1 1]      [ 1 -1 -1]     -1      -1     -        +        -
  13  Ir6-Ir7       [0 0 1]      [ 1 -1 -1]      [-1  1 -1]     -1      -1     -        +        -
  14  Ir5-Ir8    [ 0  0 -1]         [1 1 1]      [-1 -1  1]     -1      -1     -        +        -
  15  Ir6-Ir8       [0 1 0]      [ 1 -1 -1]      [-1 -1  1]     -1      -1     -        +        -
  16  Ir5-Ir7    [ 0 -1  0]         [1 1 1]      [-1  1 -1]     -1      -1     -        +        -
  17  Ir8-Ir6       [0 1 0]      [-1 -1  1]      [ 1 -1 -1]     -1      -1     -        +        -
  18  Ir5-Ir7    [ 0 -1  0]         [1 1 1]      [-1  1 -1]     -1      -1     -        +        -
  19  Ir8-Ir7       [1 0 0]      [-1 -

## 10. Mapping Between Conventions

### Equivalence of the frozen-phonon calculation

The frozen-phonon $\partial\theta/\partial Q$ is computed as:
$$\frac{\partial\theta}{\partial Q} = \frac{\theta(+Q) - \theta(0)}{Q}$$

Under the SMODES convention, $+Q$ means Group A expands, Group B contracts.
Under the notebook convention, the $+Q$ displacement (applied with their sign) means Group A contracts, Group B expands.

Since inversion maps $+Q_{\text{SMODES}} \leftrightarrow +Q_{\text{notebook}}$, and the crystal has inversion symmetry, the DFT energy and topological invariants at these two structures are **identical**. The $\partial\theta/\partial Q$ has the same magnitude; the sign just defines which direction you call "positive $Q$".

### Summary table

In [16]:
print('='*80)
print('SUMMARY: Mode 3 sign conventions')
print('='*80)
print()
print('Raw SMODES output for GM2- mode 3 (48f O):  u_i = +Q convention')
print('atomic_positions.ipynb uses:                 u_i = -Q convention (all signs flipped)')
print()
print('These are the SAME physical mode, just opposite branches of the')
print('frozen-phonon displacement. They are related by spatial inversion.')
print()
print('The notebook chose -Q because it aligns the oxygen displacements')
print('with the AIAO drive from their bridging Ir atoms.')
print()
print('Proof of equivalence:')
print('  1. Inversion maps the +Q structure exactly onto the -Q structure (Section 7)')
print('  2. Both produce the same Ir-O bond length changes (Section 6)')
print('  3. E(+Q) = E(-Q) by inversion symmetry => same DFT result')
print()
print('Either convention is correct for the frozen-phonon calculation.')
print('The QE input files in GM2-/mode3/ use the notebook (-Q) convention.')

SUMMARY: Mode 3 sign conventions

Raw SMODES output for GM2- mode 3 (48f O):  u_i = +Q convention
atomic_positions.ipynb uses:                 u_i = -Q convention (all signs flipped)

These are the SAME physical mode, just opposite branches of the
frozen-phonon displacement. They are related by spatial inversion.

The notebook chose -Q because it aligns the oxygen displacements
with the AIAO drive from their bridging Ir atoms.

Proof of equivalence:
  1. Inversion maps the +Q structure exactly onto the -Q structure (Section 7)
  2. Both produce the same Ir-O bond length changes (Section 6)
  3. E(+Q) = E(-Q) by inversion symmetry => same DFT result

Either convention is correct for the frozen-phonon calculation.
The QE input files in GM2-/mode3/ use the notebook (-Q) convention.


## 11. Cross-Check: Verify QE Input File Matches Notebook Convention

The QE input at `GM2-/mode3/Q_0.01A/input/Y2Ir2O7.scf.in` should match the $-Q$ (notebook) displaced positions.

In [17]:
# Positions from the actual QE input file (mode3, Q=0.01)
R_qe = {
    11: np.array([3.43352708, 6.36875000, 6.36875000]),
    12: np.array([9.30397292, 6.36875000, 6.36875000]),
    13: np.array([6.36875000, 6.36875000, 3.43352708]),
    14: np.array([6.36875000, 6.36875000, 9.30397292]),
    15: np.array([6.36875000, 3.43352708, 6.36875000]),
    16: np.array([6.36875000, 9.30397292, 6.36875000]),
    17: np.array([3.82125000, 6.73647292, 3.82125000]),
    18: np.array([3.82125000, 0.90602708, 3.82125000]),
    19: np.array([6.73647292, 3.82125000, 3.82125000]),
    20: np.array([0.90602708, 3.82125000, 3.82125000]),
    21: np.array([3.82125000, 3.82125000, 6.73647292]),
    22: np.array([3.82125000, 3.82125000, 0.90602708]),
}

# Our -Q displaced positions (unwrapped) should match the QE positions
# after min-image (since QE positions are wrapped).
print('QE input vs. our -Q displaced positions (min-image distance):')
print(f'{"Atom":>4}  {"d (Angstrom)":>12}')
max_err = 0
for i in range(11, 23):
    d = min_image_distance(R_minusQ[i], R_qe[i])
    max_err = max(max_err, d)
    print(f'{i:4d}  {d:.2e}')

print(f'\nMax discrepancy: {max_err:.2e}')
print('=> QE input matches -Q (notebook) convention. Confirmed.')

QE input vs. our -Q displaced positions (min-image distance):
Atom  d (Angstrom)
  11  1.65e-15
  12  1.26e-15
  13  1.35e-15
  14  2.91e-15
  15  1.65e-15
  16  1.26e-15
  17  0.00e+00
  18  2.26e-15
  19  0.00e+00
  20  2.26e-15
  21  2.26e-15
  22  4.53e-15

Max discrepancy: 4.53e-15
=> QE input matches -Q (notebook) convention. Confirmed.


### Lattice vector mapping: unwrapped $\to$ QE positions

The QE input file and the unwrapped notebook positions differ by **FCC lattice translations**. For each atom $i$:

$$\mathbf{R}^{\text{QE}}_i = \mathbf{R}^{\text{unwrapped}}_i + n_1 \mathbf{a}_1 + n_2 \mathbf{a}_2 + n_3 \mathbf{a}_3$$

where $\mathbf{a}_1 = [0, a/2, a/2]$, $\mathbf{a}_2 = [a/2, 0, a/2]$, $\mathbf{a}_3 = [a/2, a/2, 0]$ are the primitive FCC vectors and $n_1, n_2, n_3$ are integers. Since a lattice translation doesn't change the physics, these are the same structure.

In [18]:
# All QE input positions (from GM2-/mode3/Q_0.01A/input/Y2Ir2O7.scf.in)
R_qe_all = {
     1: np.array([ 5.09500000,  5.09500000,  5.09500000]),  # Y
     2: np.array([ 5.09500000,  2.54750000,  2.54750000]),  # Y
     3: np.array([ 2.54750000,  5.09500000,  2.54750000]),  # Y
     4: np.array([ 7.64250000,  7.64250000,  5.09500000]),  # Y
     5: np.array([ 0.00000000,  0.00000000,  0.00000000]),  # Ir
     6: np.array([ 5.09500000,  7.64250000,  2.54750000]),  # Ir
     7: np.array([ 7.64250000,  5.09500000,  2.54750000]),  # Ir
     8: np.array([ 2.54750000,  2.54750000,  0.00000000]),  # Ir
     9: np.array([ 3.82125000,  3.82125000,  3.82125000]),  # O (8b)
    10: np.array([ 6.36875000,  6.36875000,  6.36875000]),  # O (8b)
    11: np.array([ 3.43352708,  6.36875000,  6.36875000]),  # O (48f)
    12: np.array([ 9.30397292,  6.36875000,  6.36875000]),  # O (48f)
    13: np.array([ 6.36875000,  6.36875000,  3.43352708]),  # O (48f)
    14: np.array([ 6.36875000,  6.36875000,  9.30397292]),  # O (48f)
    15: np.array([ 6.36875000,  3.43352708,  6.36875000]),  # O (48f)
    16: np.array([ 6.36875000,  9.30397292,  6.36875000]),  # O (48f)
    17: np.array([ 3.82125000,  6.73647292,  3.82125000]),  # O (48f)
    18: np.array([ 3.82125000,  0.90602708,  3.82125000]),  # O (48f)
    19: np.array([ 6.73647292,  3.82125000,  3.82125000]),  # O (48f)
    20: np.array([ 0.90602708,  3.82125000,  3.82125000]),  # O (48f)
    21: np.array([ 3.82125000,  3.82125000,  6.73647292]),  # O (48f)
    22: np.array([ 3.82125000,  3.82125000,  0.90602708]),  # O (48f)
}

# Unwrapped positions: R_minusQ for 48f oxygens, R_eq for all others
R_unwrapped = {}
for i in sorted(R_eq):
    R_unwrapped[i] = R_minusQ[i] if i in u_GM2_mode3_smodes else R_eq[i]

# Compute lattice vector T = R_qe - R_unwrapped and decompose into primitive vectors
# T = n1*a1 + n2*a2 + n3*a3  =>  n = A_prim.T^{-1} @ T
print('Lattice vector mapping: R_QE = R_unwrapped + T')
print()
print(f'{"Atom":>4} {"Sp":>2}  {"R_unwrapped":>32}  {"R_QE":>32}  {"T = R_QE - R_unwr":>26}  {"n1":>3} {"n2":>3} {"n3":>3}  {"Residual":>10}')

max_res = 0
for i in sorted(R_eq):
    T = R_qe_all[i] - R_unwrapped[i]
    n = np.linalg.solve(A_prim.T, T)
    n_int = np.rint(n).astype(int)
    T_reconstructed = A_prim.T @ n_int
    residual = np.linalg.norm(T - T_reconstructed)
    max_res = max(max_res, residual)

    def fmt(v):
        return f'[{v[0]:9.5f}, {v[1]:9.5f}, {v[2]:9.5f}]'

    print(f'{i:4d} {species[i]:>2}  {fmt(R_unwrapped[i])}  {fmt(R_qe_all[i])}  {fmt(T)}  {n_int[0]:+3d} {n_int[1]:+3d} {n_int[2]:+3d}  {residual:.2e}')

print(f'\nMax residual: {max_res:.2e} Angstrom')
print()
print('Every T is an exact integer combination of primitive FCC vectors.')
print('The QE and unwrapped positions describe the same crystal structure.')

Lattice vector mapping: R_QE = R_unwrapped + T

Atom Sp                       R_unwrapped                              R_QE           T = R_QE - R_unwr   n1  n2  n3    Residual
   1  Y  [  5.09500,   5.09500,   5.09500]  [  5.09500,   5.09500,   5.09500]  [  0.00000,   0.00000,   0.00000]   +0  +0  +0  0.00e+00
   2  Y  [  5.09500,   2.54750,   2.54750]  [  5.09500,   2.54750,   2.54750]  [  0.00000,   0.00000,   0.00000]   +0  +0  +0  0.00e+00
   3  Y  [  2.54750,   5.09500,   2.54750]  [  2.54750,   5.09500,   2.54750]  [  0.00000,   0.00000,   0.00000]   +0  +0  +0  0.00e+00
   4  Y  [  2.54750,   2.54750,   5.09500]  [  7.64250,   7.64250,   5.09500]  [  5.09500,   5.09500,   0.00000]   +0  +0  +1  1.26e-15
   5 Ir  [  0.00000,   0.00000,   0.00000]  [  0.00000,   0.00000,   0.00000]  [  0.00000,   0.00000,   0.00000]   +0  +0  +0  0.00e+00
   6 Ir  [  0.00000,   2.54750,   2.54750]  [  5.09500,   7.64250,   2.54750]  [  5.09500,   5.09500,   0.00000]   +0  +0  +1  8.88e-16
   7 Ir

In [32]:
# Also check against +Q (SMODES) to show it does NOT match
print('QE input vs. +Q (SMODES) displaced positions (min-image distance):')
print(f'{"Atom":>4}  {"d (Angstrom)":>12}')
for i in range(11, 23):
    d = min_image_distance(R_plusQ[i], R_qe[i])
    print(f'{i:4d}  {d:.2e}')

print()
print('These are all ~0.02 Angstrom apart (= 2*Q), confirming the QE file')
print('uses the -Q convention, not the raw SMODES +Q convention.')

QE input vs. +Q (SMODES) displaced positions (min-image distance):
Atom  d (Angstrom)
  11  2.00e-02
  12  2.00e-02
  13  2.00e-02
  14  2.00e-02
  15  2.00e-02
  16  2.00e-02
  17  2.00e-02
  18  2.00e-02
  19  2.00e-02
  20  2.00e-02
  21  2.00e-02
  22  2.00e-02

These are all ~0.02 Angstrom apart (= 2*Q), confirming the QE file
uses the -Q convention, not the raw SMODES +Q convention.
